![](https://github.com/datagong/data/blob/main/datagong.png?raw=true)

© DATAGONG - Tous droits réservés - 2026

📋 **Rappels Conditions Générales d'Utilisation**

⚠️ Les Notebooks sont privés

❌ partage des Notebooks, de leur contenu, des liens, des images, …

❌ publier les Notebooks sur GitHub (public) ou tout autre outil de versionning

✅ sauvegarder les Notebooks dans votre environnement personnel et privé

✅ réutiliser les codes dans le cadre de vos projets en entreprise / projets personnels / etc.

✅ annoter / modifier les Notebooks dans votre environnement personnel et privé

# <center><u><b>Data Visualization avec Streamlit et Plotly</b></u></center>

# <b>Streamlit + Plotly — 7. Export & Partage de rapports</b>



Dans le notebook précédent, vous avez construit une page Dashboard complète avec des KPI, un graphique de tendance, un tableau filtrable et un bouton de téléchargement CSV. Votre application a tout ce qu'il faut pour visualiser les données.

Mais dans la vraie vie, vos utilisateurs ne se contentent pas de regarder un écran : ils veulent **récupérer** le graphique pour le coller dans un PowerPoint, **exporter** les données filtrées pour les retravailler dans Excel, ou encore **envoyer un lien** à un collègue pour qu'il retrouve exactement la même vue. Ce sont ces fonctionnalités d'export et de partage que nous allons mettre en place dans ce notebook.

# 0. Export d'images Plotly (PNG) avec Kaleido

Vos graphiques Plotly sont interactifs dans le navigateur, mais dès que vous voulez les intégrer dans un rapport Word, un slide ou un e‑mail, il vous faut une **image statique** (PNG, SVG…). C'est exactement ce que permet [Kaleido](https://github.com/plotly/Kaleido), un moteur d'export développé par l'équipe Plotly. Il convertit vos figures en images haute résolution directement en Python, sans avoir besoin d'ouvrir un navigateur en arrière‑plan.

## 0.1. `Figure.to_image` — générer un PNG en mémoire

La méthode [`Figure.to_image`](https://plotly.com/python-api-reference/generated/plotly.graph_objects.Figure.html#plotly.graph_objects.Figure.to_image) transforme une figure Plotly en bytes (données binaires d'une image). Vous lui passez le format souhaité (`"png"`, `"svg"`, `"jpeg"`…) et, en option, un facteur `scale` pour augmenter la résolution — `scale=2` donne par exemple une image deux fois plus nette, idéale pour l'impression ou les écrans Retina.

```python
png = fig.to_image(format="png", scale=2)
```

⚠️ Kaleido doit être installé dans votre environnement (`pip install -U kaleido`). Sans lui, `to_image` lèvera une erreur.

## 0.2. Proposer le téléchargement dans Streamlit

Une fois l'image générée, il suffit de la passer à [`st.download_button`](https://docs.streamlit.io/develop/api-reference/widgets/st.download_button) — que vous avez déjà utilisé dans le notebook précédent pour le CSV. Les paramètres sont les mêmes : `data` contient les bytes de l'image, `file_name` définit le nom du fichier téléchargé et `mime` indique le type de contenu (ici `"image/png"`).

```python
st.download_button(
    "📷 Télécharger le graphique (PNG)",
    data=png,
    file_name="graphique.png",
    mime="image/png"
)
```

💡 Vous pouvez proposer plusieurs boutons de téléchargement sur la même page — un pour le CSV, un pour le PNG. Streamlit les affichera l'un en dessous de l'autre (ou côte à côte si vous les placez dans des `st.columns`).

Exécutez la cellule ci-dessous pour ajouter l'export PNG à votre application.

In [1]:
%%writefile -a ../streamlit_app/app.py
# --- Section: Export PNG ---
import io

# 1. Générer l'image PNG à partir de la figure Plotly (nécessite kaleido)
png = fig.to_image(format="png", scale=2)

# 2. Proposer le téléchargement de l'image via un bouton Streamlit
st.download_button(
    "📷 Télécharger le graphique (PNG)",
    data=png,
    file_name="graphique.png",
    mime="image/png"
)


Appending to ../streamlit_app/app.py


# 1. Export d'un « mini‑rapport » ZIP (CSV + PNG + README)

Dans beaucoup de cas métier, un seul fichier ne suffit pas. Votre utilisateur veut récupérer **à la fois** les données filtrées, le graphique correspondant, et peut-être un petit fichier texte qui explique le contexte. Plutôt que de proposer trois boutons de téléchargement séparés, nous allons regrouper tout cela dans une archive ZIP — un seul clic, un seul fichier.

## 1.1. `zipfile` et `io.BytesIO` — créer un ZIP en mémoire

Python fournit dans sa bibliothèque standard tout ce qu'il faut pour créer des archives ZIP. Le module [`zipfile`](https://docs.python.org/3/library/zipfile.html) gère la création et la manipulation du fichier compressé, tandis que [`io.BytesIO`](https://docs.python.org/3/library/io.html#io.BytesIO) sert de tampon mémoire — concrètement, nous construisons le ZIP directement en RAM sans écrire de fichier temporaire sur le disque. C'est la bonne pratique dans une application web comme Streamlit, où vous ne voulez pas encombrer le serveur avec des fichiers provisoires.

La méthode clé est [`ZipFile.writestr`](https://docs.python.org/3/library/zipfile.html#zipfile.ZipFile.writestr) : elle permet d'écrire du contenu (chaîne de caractères ou bytes) directement dans l'archive sous un nom de fichier donné.

## 1.2. Contenu de notre archive

Dans notre cas, nous allons ajouter trois fichiers au ZIP :

Le **CSV des données filtrées**, généré comme d'habitude avec `DataFrame.to_csv(index=False)`. Le **graphique Plotly au format PNG**, produit avec `Figure.to_image` (que nous venons de voir). Et enfin un **fichier README.txt** horodaté grâce à [`time.strftime`](https://docs.python.org/3/library/time.html#time.strftime), qui donne un minimum de contexte sur l'export.

Une fois l'archive construite, nous proposons son téléchargement avec `st.download_button` en précisant le type MIME `"application/zip"`.

Exécutez la cellule ci-dessous pour ajouter l'export ZIP à votre application.

In [4]:
%%writefile -a ../streamlit_app/app.py
# --- Section: Export ZIP (rapport minimal) ---
import zipfile, time

# 1. Créer un tampon mémoire pour stocker l'archive ZIP
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w") as zf:
    # 2. Ajouter le CSV des données filtrées
    zf.writestr("data_filtre.csv", df_filtered.to_csv(index=False))
    # 3. Ajouter le graphique Plotly au format PNG
    zf.writestr("graphique.png", fig_filt.to_image(format="png", scale=2))
    # 4. Ajouter un fichier README horodaté
    zf.writestr("README.txt", "Rapport exporté depuis l'application Streamlit — "+time.strftime("%Y-%m-%d %H:%M:%S"))

# 5. Proposer le téléchargement du ZIP via Streamlit
st.download_button("📦 Exporter le rapport (.zip)", data=buf.getvalue(), file_name="rapport.zip", mime="application/zip")


Appending to ../streamlit_app/app.py


# 2. Permaliens — aller plus loin avec `st.query_params`

Dans le notebook 05, vous avez mis en place `st.query_params` pour écrire les filtres courants dans l'URL — catégories, dates. Le code est déjà en place dans `app.py` et fonctionne : quand un utilisateur modifie un filtre, l'URL se met à jour automatiquement.

Ce que nous n'avons pas encore fait, c'est exploiter cette mécanique **dans l'autre sens** : lire les paramètres de l'URL au chargement de la page pour **pré-remplir les widgets**. C'est ce qui transforme une simple URL en un vrai permalien partageable.

## 2.1. Initialiser les widgets depuis l'URL

Le principe est simple : au chargement de la page, vous lisez `st.query_params` pour récupérer les valeurs éventuellement présentes dans l'URL, puis vous les utilisez comme valeurs par défaut de vos widgets. Si l'URL ne contient aucun paramètre, vos valeurs par défaut habituelles s'appliquent.

```python
params = st.query_params

# Lire la catégorie dans l'URL, sinon prendre la première disponible
default_cat = params.get("categorie", cats[0])

# Utiliser cette valeur comme sélection par défaut
cat = st.selectbox("Catégorie", options=cats, index=cats.index(default_cat))
```

Grâce à cette boucle lecture → widget → écriture, un collègue qui ouvre votre permalien retrouvera exactement les mêmes filtres appliqués, et pourra les modifier à son tour — l'URL suivra.

💡 Pensez à gérer le cas où la valeur dans l'URL n'existe plus dans vos données (par exemple, une catégorie supprimée). Un simple `if default_cat in cats` avant de l'utiliser suffit à éviter une erreur.

## 2.2. Proposer un lien copiable dans l'interface

Pour que vos utilisateurs n'aient pas à fouiller dans la barre d'adresse du navigateur, vous pouvez afficher le permalien directement dans votre application. Un simple `st.text_input` en lecture seule fait très bien l'affaire — l'utilisateur n'a plus qu'à cliquer et copier.

```python
st.text_input("🔗 Permalien", value=f"http://localhost:8501/?categorie={cat}", disabled=True)
```

Combiné à l'écriture dans `st.query_params` que vous avez déjà codée dans le notebook 05, cela rend votre application aussi partageable qu'un Google Sheet avec ses filtres appliqués.

# <font color='#ff7373'><b>Félicitations !</b></font>

Votre application sait désormais exporter des graphiques en PNG, générer un mini-rapport ZIP complet, et partager un état précis via un permalien. Ce sont des fonctionnalités que vos utilisateurs métier apprécieront au quotidien.

Dans le prochain notebook, place au **capstone** : vous assemblerez toutes les briques vues depuis le début de la formation dans une application complète.

# <center><font color='#3b4859'><u>![](https://github.com/datagong/data/blob/main/mini%20datagong%202.png?raw=true)</u></font></center>